# 03 · Evaluation, calibration and the leakage comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rishabh-bgp/brain-tumor-mri-classification/blob/main/notebooks/03_evaluation_and_calibration.ipynb)

Turns saved logits into the single prediction file (`results/predictions/*_test_predictions.csv`), fits temperature scaling and the *uncertain* threshold on validation, and derives every reported metric, table, figure and the web demo's statistics from that file.

*Rishabh Aryan · M.Tech (AI & Data Science), IIIT Bhagalpur. Research and education code, not a medical device.*

In [ ]:
# --- Setup: works locally (repo checkout) and on Google Colab ---
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("brain-tumor-mri-classification"):
        subprocess.run(["git", "clone", "-q", "https://github.com/Rishabh-bgp/brain-tumor-mri-classification.git"], check=True)
    os.chdir("brain-tumor-mri-classification")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "kagglehub", "imagehash", "captum",
                    "onnx", "onnxruntime", "scikit-image"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("working directory:", os.getcwd())

In [ ]:
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU (Colab: Runtime > Change runtime type > GPU for faster training)")

In [ ]:
subprocess.run([sys.executable, "scripts/make_predictions.py"], check=True)
subprocess.run([sys.executable, "scripts/make_report.py"], check=True)
import json, pandas as pd
cal = json.load(open("results/predictions/calibration.json"))
print(json.dumps(cal, indent=1))

In [ ]:
pred = pd.read_csv("results/predictions/clean_test_predictions.csv")
display(pred.head())
m = json.load(open("results/metrics.json"))
rows = []
for run in ("clean", "naive"):
    for k, v in m[run].items():
        rows.append({"model": run, "subset": k, "n": v["n"], "accuracy": v["accuracy"], "acc 95% CI": v["accuracy_ci95"],
                     "macro_F1": v["macro_f1"], "ROC-AUC": v.get("roc_auc_macro_ovr"), "ECE": v["ece_calibrated"]})
display(pd.DataFrame(rows).round(4))

In [ ]:
full = m["clean"]["full_test"]
display(pd.DataFrame(full["per_class"]).T.round(4))
print("confusion matrix (rows = true, cols = predicted; glioma, meningioma, notumor, pituitary):")
print(pd.DataFrame(full["confusion_matrix"], index=["glioma", "meningioma", "notumor", "pituitary"], columns=["glioma", "meningioma", "notumor", "pituitary"]))
print("uncertain rate:", round(full["uncertain_rate"], 4), "| accuracy confident:", full["accuracy_confident"], "| accuracy uncertain:", full["accuracy_uncertain"])

### Leakage by class: Testing images with vs without a strict near-duplicate in the original Training folder

In [ ]:
lb = pd.DataFrame(m["leakage_by_class"]).T
lb.index = pd.MultiIndex.from_tuples([tuple(i.split("|")) for i in lb.index], names=["model", "class", "subset"])
display(lb.unstack(["model", "subset"]))
subprocess.run([sys.executable, "scripts/audit_augmented.py"], check=True)  # files named *-aug-* (augmented meningiomas)

### Robustness check: a looser link rule (no retraining)
How many more Testing images would a rule that also links Hamming 7–10 with correlation ≥ 0.95 tie to our train/validation images, and what is the accuracy without them?

In [ ]:
print(json.dumps(m["sensitivity_looser_rule"], indent=1))
sd = m["clean"]["test_with_strict_train_dup"]
print(sd["macro_f1_note"]); print("macro-F1 over present classes", sd["present_classes"], "=", round(sd["macro_f1_present_classes"], 4))

In [ ]:
from IPython.display import Image, display
for f in ("confusion_matrix", "roc_curves", "reliability"):
    display(Image(f"results/figures/{f}.png", width=480))